In [47]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options as ChromeOptions 
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import re

import requests
import csv


# 오늘의 단어

In [48]:

def get_today_words_from_naver():
    
    options = ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument("user-agent=Mozilla/5.0")

    service = ChromeService(executable_path="C:\\Users\\서민경\\Desktop\\vocab\\chromedriver-win64\\chromedriver.exe")
    driver = webdriver.Chrome(service=service, options=options)
    try:
        url = "https://search.naver.com/search.naver?query=오늘의+단어"
        driver.get(url)
        time.sleep(2)  # 로딩 대기

        words = []
        
        # 모든 단어 항목 선택
        word_links = driver.find_elements(By.CSS_SELECTOR, "a.word")

        for link in word_links:
            try:
                word = link.find_element(By.TAG_NAME, "strong").text.strip()
                # 뜻은 link 다음에 나오는 <span class="mean"> 요소임
                meaning = link.find_element(By.XPATH, "./following-sibling::span[@class='mean']").text.strip()
                words.append((word, meaning))
            except Exception as e:
                print(f"❗ 오류 발생: {e}")
                continue

        return words

    finally:
        driver.quit()

# 실행
word_list = get_today_words_from_naver()
print(f"총 {len(word_list)}개 단어 추출됨.")
for word, meaning in word_list:
    print(f"{word} - {meaning}")


총 5개 단어 추출됨.
eat - (음식·밥 등을) 먹다
govern - (국가·국민을) 통치하다[다스리다]
glisten - 반짝이다, 번들거리다
lengthen - 길어지다; 늘이다
constructive - 건설적인


# TEPS 90

In [49]:

def get_teps():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')

    driver = webdriver.Chrome(service=Service(), options=options)
    driver.get("https://blog.naver.com/lblucy/223865867643")

    # iframe 내부로 진입
    driver.switch_to.frame("mainFrame")
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    driver.quit()

    # 원하는 span만 추출
    spans = soup.find_all('span', class_='se-fs- se-ff-')
    texts = [span.get_text(strip=True) for span in spans if span.get_text(strip=True)]

    results = []
    temp = {}

    for text in texts:
        if '의미 :' in text:
            temp['의미'] = text.replace('의미 :', '').strip()
        elif '예문 :' in text:
            temp['예문'] = text.replace('예문 :', '').strip()
        elif '뜻 :' in text:
            temp['뜻'] = text.replace('뜻 :', '').strip()
            if '단어' in temp and '의미' in temp and '예문' in temp:
                results.append(temp)
                temp = {}
        else:
            # '단어'는 의미, 예문, 뜻이 아닌 일반 텍스트로 간주
            temp = {'단어': text}

    return results

# 출력
for entry in get_teps():
    print(f"단어: {entry.get('단어')}")
    print(f"의미: {entry.get('의미')}")
    print(f"예문: {entry.get('예문')}")
    print(f"뜻: {entry.get('뜻')}")
    print('-' * 50)


단어: ordinary
의미: 평범한
예문: The meal was very ordinary
뜻: 식사는 아주 평범했다.
--------------------------------------------------
단어: envious
의미: 부러워하는
예문: Everyone is so envious of her.
뜻: 모든 사람들이 그녀를 몹시 부러워한다.
--------------------------------------------------
단어: prospective
의미: 유망한
예문: a prospective buyer
뜻: 장래의 구매자
--------------------------------------------------
단어: shake up
의미: 충격을 주다
예문: She shakes him up
뜻: 그녀는 그에게 충격을 준다.
--------------------------------------------------
단어: clog
의미: 막다
예문: The narrow streets were clogged with traffic.
뜻: 좁은 도로가 차들로 꽉 막혀 있었다.
--------------------------------------------------
단어: tricky
의미: 까다로운
예문: a tricky situation
뜻: 곤란한 상황
--------------------------------------------------
단어: green light
의미: 허가, 승인
예문: The government has decided to give the green light to the plan.
뜻: 정부가 그 계획을 승인하기로 결정했다.
--------------------------------------------------
단어: alternate
의미: 번갈아 나오는
예문: alternate layers of fruit and cream
뜻: 과일과 크림이 번갈아 층층이 놓인 것
----------------

# TEPS 딕셔너리 형태로 변환

In [50]:

def trans_teps_data(text_lines):
    """
    text_lines: 블로그에서 추출한 한 줄짜리 문자열 리스트
                [단어, '의미 : xxx', '예문 : yyy', '뜻 : zzz', ...] 패턴으로 반복된다고 가정.
    return     : [{'word': ..., 'meaning': ..., 'example': ..., 'example_meaning': ...}, ...]
    """
    parsed = []
    i = 0

    while i < len(text_lines):
        word = text_lines[i].strip()

        # 최소 3줄 더 있어야 의미/예문/뜻이 붙어 있다고 판단
        if (i + 3 < len(text_lines)
            and text_lines[i + 1].startswith("의미")
            and text_lines[i + 2].startswith("예문")
            and text_lines[i + 3].startswith("뜻")):

            meaning = re.sub(r"^의미\s*:", "", text_lines[i + 1]).strip()
            example = re.sub(r"^예문\s*:", "", text_lines[i + 2]).strip()
            example_meaning = re.sub(r"^뜻\s*:", "", text_lines[i + 3]).strip()

            parsed.append({
                "word": word,
                "meaning": meaning,
                "example": example,
                "example_meaning": example_meaning
            })

            i += 4   # 다음 단어 블록으로 이동
        else:
            # 패턴이 깨진 경우 한 줄만 넘기고 계속 탐색
            i += 1

    return parsed


raw_data = get_teps()  # 이미 dict 형태로 되어 있음

for entry in raw_data:
    print(entry)



{'단어': 'ordinary', '의미': '평범한', '예문': 'The meal was very ordinary', '뜻': '식사는 아주 평범했다.'}
{'단어': 'envious', '의미': '부러워하는', '예문': 'Everyone is so envious of her.', '뜻': '모든 사람들이 그녀를 몹시 부러워한다.'}
{'단어': 'prospective', '의미': '유망한', '예문': 'a prospective buyer', '뜻': '장래의 구매자'}
{'단어': 'shake up', '의미': '충격을 주다', '예문': 'She shakes him up', '뜻': '그녀는 그에게 충격을 준다.'}
{'단어': 'clog', '의미': '막다', '예문': 'The narrow streets were clogged with traffic.', '뜻': '좁은 도로가 차들로 꽉 막혀 있었다.'}
{'단어': 'tricky', '의미': '까다로운', '예문': 'a tricky situation', '뜻': '곤란한 상황'}
{'단어': 'green light', '의미': '허가, 승인', '예문': 'The government has decided to give the green light to the plan.', '뜻': '정부가 그 계획을 승인하기로 결정했다.'}
{'단어': 'alternate', '의미': '번갈아 나오는', '예문': 'alternate layers of fruit and cream', '뜻': '과일과 크림이 번갈아 층층이 놓인 것'}
{'단어': 'as long as', '의미': '~ 하는 한', '예문': 'Stay as long as you like', '뜻': '있고 싶은 대로 오래 있어'}
{'단어': 'file', '의미': '제기하다', '예문': 'to file for divorce', '뜻': '이혼 소송을 제기하다'}
{'단어': 'act up', '의미': '말을 안 듣고 버

In [55]:
import re

def get_variants(word):
    return list(filter(None, [
        word,
        word + 's',
        word + 'ed',
        word + 'ing',
        word + 'es',
        word + 'd' if word.endswith('e') else ''
    ]))

dict_result = []

for item in raw_data:
    word = item.get("단어")
    meaning = item.get("뜻")
    example = item.get("예문")
    example_meaning = item.get("해석", "")

    pattern = r'\b' + re.escape(word) + r'\b'
    replaced_example = re.sub(pattern, "_________", example, flags=re.IGNORECASE)

    entry = {
        "word": word,
        "meaning": meaning,
        "example": replaced_example,
    }

    if example_meaning:
        entry["example_meaning"] = example_meaning

    dict_result.append(entry)

for entry in dict_result:
    print(entry)

{'word': 'ordinary', 'meaning': '식사는 아주 평범했다.', 'example': 'The meal was very _________'}
{'word': 'envious', 'meaning': '모든 사람들이 그녀를 몹시 부러워한다.', 'example': 'Everyone is so _________ of her.'}
{'word': 'prospective', 'meaning': '장래의 구매자', 'example': 'a _________ buyer'}
{'word': 'shake up', 'meaning': '그녀는 그에게 충격을 준다.', 'example': 'She shakes him up'}
{'word': 'clog', 'meaning': '좁은 도로가 차들로 꽉 막혀 있었다.', 'example': 'The narrow streets were clogged with traffic.'}
{'word': 'tricky', 'meaning': '곤란한 상황', 'example': 'a _________ situation'}
{'word': 'green light', 'meaning': '정부가 그 계획을 승인하기로 결정했다.', 'example': 'The government has decided to give the _________ to the plan.'}
{'word': 'alternate', 'meaning': '과일과 크림이 번갈아 층층이 놓인 것', 'example': '_________ layers of fruit and cream'}
{'word': 'as long as', 'meaning': '있고 싶은 대로 오래 있어', 'example': 'Stay _________ you like'}
{'word': 'file', 'meaning': '이혼 소송을 제기하다', 'example': 'to _________ for divorce'}
{'word': 'act up', 'meaning': '아이들이 말을 안 듣기

In [52]:

with open("teps_words.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["단어", "예문", "의미", "뜻"])
    writer.writeheader()
    for entry in raw_data:
        writer.writerow(entry)


# 오늘의 단어 추가

In [53]:


# 웹페이지 URL
url = "https://englishparadise.tistory.com/61"

# 웹페이지 요청
response = requests.get(url)
response.raise_for_status()  # 요청이 성공했는지 확인

# HTML 파싱
soup = BeautifulSoup(response.text, "html.parser")

# 본문 내용 추출
content_div = soup.find("div", class_="tt_article_useless_p_margin")
if not content_div:
    raise ValueError("본문 내용을 찾을 수 없습니다.")

# 텍스트 추출 및 전처리
text = content_div.get_text(separator="\n")
text = text.replace("\xa0", " ").strip()

# 불필요한 서두 제거
start_phrase = "영단어 정리"
start_index = text.find(start_phrase)
if start_index == -1:
    raise ValueError("영단어 정리 부분을 찾을 수 없습니다.")
text = text[start_index + len(start_phrase):].strip()

# 단어와 의미 분리
lines = text.split("\n")
word_meanings = []
for line in lines:
    if " - " in line:
        word, meaning = line.split(" - ", 1)
        word_meanings.append((word.strip(), meaning.strip()))

#결과 출력력
for word, meaning in word_meanings:
    print(f"{word} - {meaning}")



cultivate - 함양, 배양하다, 기르다
coincide - 일치하다, 동시에 일어나다
timely - 시기적절한
artfully - 교묘하게
vulnerability - 취약성
recolonize - 다시 대량 서식하다
at best - 기껏해야
modify - 수정하다
condition - 조건
exclaim - 외치다, 소리치다
entitlement - 재정지원, 혜택
tracing - 모사, 투사
disquieting - 불안한, 불안하게 하는
attain - 얻다
accordance with - a에 따라, a 부합되게
return - 수익
with - 맞물리다, 동시에 일어나다
sound - 타당한
occasional - 이따금
cate - 유포하다
companionship - 동료 관계
prescribe - 규정하다, 지시하다, 처방하다
reflection - 성찰
exclusive - 독점적인
scatter - 흩어지게 하다
circulate - 배포하다, 돌리다
accommodations - 숙박 시설
famine - 기근
depart - 벗어나다, 출발하다
residual - 잔여의, 나머지의
inquire - 문의하다
enterprising - 진취적인
spatial - 공간의
institutional - 제도적인, 제도상의
acquisition - 습득, 획득
superior - 뛰어난, 우수한, 우월한
draw - 이끌다, 당기다
taxation - 과세
hard-nosed - 엄격한, 냉철한
multiply - 배가하다, 늘리다
be accustomed to - 익숙하다
outstanding - 뛰어난
reproduce - 재현하다
informed - 사실 이해에 입각한, 정보에 근거한
ongoing - 지속적인
pretend - 가장하다
document - 기록하다
renewable - 재생 가능한
singular - 기묘한
operant - 조작에 작용하는
utilize - 활용하다
novel - 새로운, 참신한
flawles

# 오늘의 단어 20 CSV

In [54]:
with open("today_words20.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["word", "meaning"])
    for word, meaning in word_meanings:
        writer.writerow([word, meaning])